# 01 · Supervised fraud classifier (XGBoost)

Learns the laundering that was **already caught**: the labelled payments in the IBM AML data.
Accurate on known patterns, blind to new ones — the Isolation Forest in `02` covers that side
(`docs/design-decisions.md` §3).

**Data.** IBM *Transactions for Anti-Money-Laundering*, `HI-Small_Trans.csv` — about 5M
synthetic payments, 0.1% laundering. Each payment becomes a DEBIT row for the payer and a
CREDIT row for the payee (`sentinel_features.py`), because Sentinel scores from one customer's
side.

**Split.** By time: earlier days train, later days test. A random split would leak a
laundering chain's later hops into training.

**Metric.** PR-AUC, not accuracy — predicting "never laundering" is 99.9% accurate.

In [ ]:
import json, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent))    # ml/, where sentinel_features.py lives
import sentinel_features as sf

sf.MODELS.mkdir(exist_ok=True)
N = len(sf.FEATURES)
rows = sf.load_rows()        # first run builds the features (~2 min), later runs read the cache
train, test, cutoff = sf.time_split(rows)
print(f"{len(rows):,} rows; train up to {cutoff}, {len(train):,} rows, {train.label.mean():.4%} laundering;"
      f" test {len(test):,} rows, {test.label.mean():.4%} laundering")

In [ ]:
import xgboost as xgb

y_train = train.label.values
model = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    # ~1 positive per 1,000: weight them up rather than throw 99.9% of the data away.
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    tree_method="hist", n_jobs=-1, random_state=42,
)
model.fit(sf.matrix(train), y_train)

In [ ]:
from sklearn.metrics import average_precision_score, precision_recall_curve

y_test = test.label.values
p_test = model.predict_proba(sf.matrix(test))[:, 1]
pr_auc = average_precision_score(y_test, p_test)
print(f"PR-AUC {pr_auc:.3f}   (a random ranking scores {y_test.mean():.4f})")

precision, recall, thresholds = precision_recall_curve(y_test, p_test)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(recall, precision)
ax.set(xlabel="recall", ylabel="precision", title=f"XGBoost on the later days - PR-AUC {pr_auc:.3f}")
ax.grid(alpha=.3)
plt.show()

pd.Series(model.feature_importances_, index=sf.FEATURES).sort_values().plot.barh(
    figsize=(6, 4), title="Feature importance (gain share)")
plt.show()

## Choosing the threshold

The threshold is a business decision, not a modelling one: it sets how many alerts the
investigation team gets per true case. The table shows the trade; the chosen value goes into
`scoring-service` config (`sentinel.scoring.models.classifier-threshold`), not into the model.

In [ ]:
table = pd.DataFrame({"threshold": thresholds, "precision": precision[:-1], "recall": recall[:-1]})
picks = [table[table.precision >= target].iloc[0] for target in (0.2, 0.3, 0.4, 0.5, 0.6)
         if (table.precision >= target).any()]
pd.DataFrame(picks).assign(alerts_per_true_case=lambda t: 1 / t.precision).round(3)

## Export to ONNX

`scoring-service` runs the `.onnx` file with ONNX Runtime inside the JVM. The input is one
float tensor named `features`, in the order of `sentinel_features.FEATURES`.

In [ ]:
import onnx
import onnxruntime as ort

def attach(model_proto, **metadata):
    """Metadata travels inside the .onnx file, so it cannot drift from the model it describes."""
    for key, value in metadata.items():
        entry = model_proto.metadata_props.add()
        entry.key, entry.value = key, json.dumps(value)
    return model_proto

def save(model_proto, name):
    path = sf.MODELS / f"{name}.onnx"
    onnx.save(model_proto, path)
    (sf.MODELS / f"{name}.features.json").write_text(json.dumps(sf.FEATURES, indent=2) + "\n")
    print(f"wrote {path.name} ({path.stat().st_size / 1024:.0f} KB) and {name}.features.json")
    return ort.InferenceSession(str(path))

# A handful of real rows, fraud included, that scoring-service's parity test replays through
# ONNX Runtime in Java and compares with what Python got here.
parity_rows = pd.concat([test[test.label == 1].head(4), test[test.label == 0].head(4)])
parity_inputs = sf.matrix(parity_rows)

In [ ]:
from onnxmltools import convert_xgboost
from onnxmltools.convert.common.data_types import FloatTensorType

proto = convert_xgboost(model, initial_types=[("features", FloatTensorType([None, N]))],
                        target_opset=15)
parity_expected = model.predict_proba(parity_inputs)[:, 1]
attach(proto, features=sf.FEATURES, pr_auc=round(pr_auc, 4),
       parity={"inputs": parity_inputs.tolist(), "probability": parity_expected.tolist()})
session = save(proto, "fraud_xgboost")

# Same scores from ONNX Runtime as from XGBoost, on the whole test set.
sample = sf.matrix(test.sample(200_000, random_state=1))
onnx_p = session.run(None, {"features": sample})[1][:, 1]
print("outputs:", [o.name for o in session.get_outputs()])
print("max |onnx - xgboost| =", np.abs(onnx_p - model.predict_proba(sample)[:, 1]).max())